In [ ]:
!cp "/content/drive/MyDrive/WSDM Paper/Data/researchers.csv" .

In [ ]:
!pip install fuzzywuzzy

In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import pandas as pd
import numpy as np
from collections import OrderedDict
import scipy.stats as stats

/usr/local/lib/python3.12/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [ ]:
def extractLastNames(names):
    lastNames = []
    for name in names:
      parts = name.strip().lower().split()
      lastNames.append(parts[-1].strip())
    return lastNames

In [ ]:
thresholds = [60, 70, 80, 90]

In [ ]:
df = pd.read_csv("researchers.csv")
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Region,Co-author count (Google Scholar),Co-authors' names (Google Scholar),Co-authors' names (OpenAlex),Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral)
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,East/South-East Asia,3,K Ranjeet/Neelesh Dahanukar/Rajeev Raghavan,B Madhusoodana Kurup/Balakrishna Madhusoodana ...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,East/South-East Asia,3,Jamroon Srichaichana/Dokrak Marod/Hee Han,Akbar I. Inamdar/Arjun Magotra/Atchara Teerawa...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,East/South-East Asia,3,A.R.S.B. Athauda/Saman Athauda/M Pagthinathan/...,A. R. S. B. Athauda/Clement R. de Cruz/E. Pows...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,East/South-East Asia,3,Poulomi Dey/Jiwan Paudel/Ashish Chaudhary,Akshay Chaudhary/Alberto Sanz-Cobeña/Alice Min...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,East/South-East Asia,5,S. Venkatesan/T.Uma Maheswari/S Kumar/R. Sudha...,A. Bedoui/A. Deepak/A. K. Ramasamy/A. Kanakala...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-


In [ ]:
df = df[df["Reality (DeepSeek R1)"] != "hypothetical"]
df = df[df["Co-authors' names (DeepSeek R1)"].notna()]
df = df[df["Co-authors' names (OpenAlex)"].notna()]

In [ ]:
len(df)

1349

In [ ]:
df["Match Count 60%"] = 0
df["Match Count 70%"] = 0
df["Match Count 80%"] = 0
df["Match Count 90%"] = 0

In [ ]:
for index, row in df.iterrows():
    llmNames = row["Co-authors' names (DeepSeek R1)"].split('/')
    refNames = row["Co-authors' names (OpenAlex)"].split('/')

    llmNames = extractLastNames(llmNames)
    refNames = extractLastNames(refNames)

    for threshold in thresholds:
        count = 0
        refNamesCopy = refNames.copy()

        for name in llmNames:
            if not refNamesCopy:
                break

            bestRatio = 0
            bestIndex = -1
            for i, candidate in enumerate(refNamesCopy):
                currentRatio = fuzz.ratio(name, candidate)
                if currentRatio >= threshold and currentRatio > bestRatio:
                    bestRatio = currentRatio
                    bestIndex = i

            if bestIndex != -1:
                count += 1
                del refNamesCopy[bestIndex]

        colName = "Match Count " + str(threshold) + "%"
        df.at[index, colName] = count

In [ ]:
df.head()

,Name,Google Scholar ID,OpenAlex URL,Citation Count,Publication Count,Email Domain,Affiliation,Field,Subfield,Citation Level,...,Co-authors' names (DeepSeek R1),Reality (DeepSeek R1),Co-authors' names (Llama 4 Scout),Reality (Llama 4 Scout),Co-authors' names (Mixtral),Reality (Mixtral),Match Count 60%,Match Count 70%,Match Count 80%,Match Count 90%
0,Kuttanelloor Roshni,xvwC5SMAAAAJ,https://openalex.org/a5049943816,163,29,@cusat.ac.in,"Post Doctoral Fellow, School of Environmental ...","Agriculture, Fisheries & Forestry",Fisheries,Low,...,Shashi Bhushan/Vinay Kumar Vase/Achamveetil Go...,-,NaN,-,S. Sree Ranjani/S. Jegadhesan/R. Anbazhagan/K ...,-,1,1,1,0
1,Siriluck Thammanu,H6zh8jwAAAAJ,https://openalex.org/A5090976916,120,8,@mail.forest.go.th,"Royal Forest Department, Thailand","Agriculture, Fisheries & Forestry",Forestry,Low,...,Harald Vacik/Eakaphong Ashraf/Shinya Numata,-,NaN,-,Pattarapong Kongcharoen/Kitiyos Srisuksanti/Su...,-,1,0,0,0
2,M.S.M. Nafees,eQr4PLYAAAAJ,https://openalex.org/a5079244643,113,15,@esn.ac.lk,Senior Lecturer,"Agriculture, Fisheries & Forestry",Dairy & Animal Science,Low,...,Qamar Bilal/Aisha Khatoon/Muhammad Tahir,-,NaN,-,Muhammad Farrukh Shahid/Abdul Rauf/Muhammad Im...,-,0,0,0,0
3,Aman Kumar Gupta,cab6DuAAAAAJ,https://openalex.org/a5101497736,122,17,NaN,Banaras Hindu University,"Agriculture, Fisheries & Forestry",Agronomy & Agriculture,Low,...,Rajendra Prasad/Anil Kumar Singh/Shiv Datt Sharma,-,NaN,-,Surjeet Singh/Jaspreet Singh/Amrik S. Basra,-,3,2,1,1
4,Seralathan Kamala‐Kannan,ERXuIzUAAAAJ,https://openalex.org/A5075977093,158,178,@annamalaiuniversity.ac.in,"Department of Horticulture, Faculty of Agricul...","Agriculture, Fisheries & Forestry",Horticulture,Low,...,K. Prabakar/R. Samiyappan/A. Nithya/M. Senthil...,-,NaN,-,Ramalingam Palanisamy/Arulmozhi Marimuthu/Siva...,-,5,5,2,2


In [ ]:
df["DNE 60%"] = 0
df["DNE 70%"] = 0
df["DNE 80%"] = 0
df["DNE 90%"] = 0

for index, row in df.iterrows():

    N = len(row["Co-authors' names (Google Scholar)"].split("/"))
    R = len(row["Co-authors' names (OpenAlex)"].split("/"))

    for threshold in [60, 70, 80, 90]:
        match_col = f"Match Count {threshold}%"
        DNE_col = f"DNE {threshold}%"

        denom = max(1, min(N, R))
        DNE = min(row[match_col] / denom, 1.0)

        df.at[index, DNE_col] = DNE

/tmp/ipython-input-2300685900.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, DNE_col] = DNE
/tmp/ipython-input-2300685900.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, DNE_col] = DNE
/tmp/ipython-input-2300685900.py:18: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.3333333333333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[index, DNE_col] = DNE
/tmp/ipython-input-2300685900.py:18: FutureWarning: Setting an item of incompatible dtype i

In [ ]:
print("Mean DNE", np.mean(df["DNE 60%"]))
print("Std DNE", df["DNE 60%"].std())

Mean DNE 0.5361817700586036
Std DNE 0.3473313795306496


In [ ]:
fields = list(df["Field"].unique())
fields

['Agriculture, Fisheries & Forestry',
 'Biology',
 'Built Environment & Design',
 'Clinical Medicine',
 'Earth & Environmental Sciences',
 'Economics & Business',
 'Engineering',
 'Information & Communication Technologies',
 'Mathematics & Statistics',
 'Physics & Astronomy']

In [ ]:
metrics = ["DNE 60%", "DNE 70%", "DNE 80%", "DNE 90%"]

levels = ['High', 'Low']

In [ ]:
regions = list(df["Region"].unique())
regions

['East/South-East Asia',
 'Europe',
 'Middle East',
 'North Africa',
 'North America',
 'Oceanic',
 'South/Central America',
 'Sub-Saharan Africa']

# Mean DNE without disaggregation

In [ ]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
      print(metric, "*****", sf[metric].mean())

Agriculture, Fisheries & Forestry
DNE 60% ***** 0.5844374619846602
DNE 70% ***** 0.3417574743037279
DNE 80% ***** 0.26207240301830037
DNE 90% ***** 0.21565677434815975
Biology
DNE 60% ***** 0.5671997966404324
DNE 70% ***** 0.35299860684306844
DNE 80% ***** 0.27213572308022205
DNE 90% ***** 0.23204330985725702
Built Environment & Design
DNE 60% ***** 0.41676642266927216
DNE 70% ***** 0.23152939556075555
DNE 80% ***** 0.18796541266981664
DNE 90% ***** 0.16417086467448253
Clinical Medicine
DNE 60% ***** 0.6522037637702728
DNE 70% ***** 0.4365562256723534
DNE 80% ***** 0.31610488449973057
DNE 90% ***** 0.2640252471307997
Earth & Environmental Sciences
DNE 60% ***** 0.5952059712017808
DNE 70% ***** 0.39044461775186556
DNE 80% ***** 0.3223558127246707
DNE 90% ***** 0.27761888706690874
Economics & Business
DNE 60% ***** 0.4401240914668132
DNE 70% ***** 0.24049638642357432
DNE 80% ***** 0.1780270302469651
DNE 90% ***** 0.15929434831838407
Engineering
DNE 60% ***** 0.5129530411689579
DNE 70% **

In [ ]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    print(metric, "*****", sf[metric].mean())

East/South-East Asia
DNE 60% ***** 0.6161788475293921
DNE 70% ***** 0.4507316606657595
DNE 80% ***** 0.36624756461605473
DNE 90% ***** 0.31621300935664093
Europe
DNE 60% ***** 0.533527626068095
DNE 70% ***** 0.33964301127629737
DNE 80% ***** 0.2625665997616754
DNE 90% ***** 0.22093034768678405
Middle East
DNE 60% ***** 0.48813001787324417
DNE 70% ***** 0.2598527737291654
DNE 80% ***** 0.17372033901381195
DNE 90% ***** 0.1356614870597617
North Africa
DNE 60% ***** 0.4782811752146127
DNE 70% ***** 0.23165374806680847
DNE 80% ***** 0.17058421171521268
DNE 90% ***** 0.1389767473831632
North America
DNE 60% ***** 0.5596520400039503
DNE 70% ***** 0.34871750818251
DNE 80% ***** 0.28714899184502657
DNE 90% ***** 0.25669263632876843
Oceanic
DNE 60% ***** 0.5985782006831045
DNE 70% ***** 0.424175746039595
DNE 80% ***** 0.355763763330727
DNE 90% ***** 0.3163556614647008
South/Central America
DNE 60% ***** 0.5121057310290127
DNE 70% ***** 0.32058404356548287
DNE 80% ***** 0.2630826352544565
DNE 90

# Mean DNE disaggregated by Citation Level

In [ ]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

Agriculture, Fisheries & Forestry
High
DNE 60% ***** 0.7732663122336866
DNE 70% ***** 0.48859506015744564
DNE 80% ***** 0.39820941340924815
DNE 90% ***** 0.3358233966551748
Low
DNE 60% ***** 0.36905455466936443
DNE 70% ***** 0.1742708529393312
DNE 80% ***** 0.10679112554112555
DNE 90% ***** 0.07859172077922078
Biology
High
DNE 60% ***** 0.7211867929870949
DNE 70% ***** 0.4824209452543067
DNE 80% ***** 0.3806265257355094
DNE 90% ***** 0.32056241794876394
Low
DNE 60% ***** 0.38193419166085407
DNE 70% ***** 0.19728735594204744
DNE 80% ***** 0.1416077261355794
DNE 90% ***** 0.12554375793466266
Built Environment & Design
High
DNE 60% ***** 0.5187229493365068
DNE 70% ***** 0.29662664318413656
DNE 80% ***** 0.23855166925974325
DNE 90% ***** 0.20615518312014913
Low
DNE 60% ***** 0.29441859066859066
DNE 70% ***** 0.1534126984126984
DNE 80% ***** 0.12726190476190477
DNE 90% ***** 0.11378968253968254
Clinical Medicine
High
DNE 60% ***** 0.7649660693511392
DNE 70% ***** 0.5721083737175683
DNE 80% 

In [ ]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for level in levels:
    print(level)
    hf = sf[sf["Citation Level"] == level]
    for metric in metrics:
      print(metric, "*****", hf[metric].mean())

East/South-East Asia
High
DNE 60% ***** 0.8061083690745411
DNE 70% ***** 0.6367739140682086
DNE 80% ***** 0.5506997051371476
DNE 90% ***** 0.4871555189033344
Low
DNE 60% ***** 0.39079581529581525
DNE 70% ***** 0.22996151996151995
DNE 80% ***** 0.14736435786435786
DNE 90% ***** 0.11336123136123134
Europe
High
DNE 60% ***** 0.7202246653136067
DNE 70% ***** 0.495527088356665
DNE 80% ***** 0.4124176511055947
DNE 90% ***** 0.3498566566497591
Low
DNE 60% ***** 0.31571441361499814
DNE 70% ***** 0.15777825468253517
DNE 80% ***** 0.08774037319376961
DNE 90% ***** 0.07051632056331303
Middle East
High
DNE 60% ***** 0.6553210406166545
DNE 70% ***** 0.3842905611806744
DNE 80% ***** 0.2708069575802568
DNE 90% ***** 0.2148038540961698
Low
DNE 60% ***** 0.29765923246935905
DNE 70% ***** 0.11808820574643357
DNE 80% ***** 0.06311533052039382
DNE 90% ***** 0.045499296765119546
North Africa
High
DNE 60% ***** 0.6327492985697158
DNE 70% ***** 0.3442584197949365
DNE 80% ***** 0.2458552866461655
DNE 90% ****

# Mean Comparison

In [ ]:
lowCitedDNE = df[df["Citation Level"] == "Low"]["DNE 60%"].to_list()
highlyCitedDNE = df[df["Citation Level"] == "High"]["DNE 60%"].to_list()

In [ ]:
print("Low cited Avg DNE:", np.mean(lowCitedDNE))
print("highly cited Avg DNE:", np.mean(highlyCitedDNE))

Low cited Avg DNE: 0.3492104097016634
highly cited Avg DNE: 0.6956724497037408


In [ ]:
t_stat, p_value = stats.ttest_ind(highlyCitedDNE, lowCitedDNE, alternative='greater')

In [ ]:
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_value}")

T-statistic: 21.0409241493047
P-value: 1.6471614768871988e-85


## T-Test

In [ ]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 1.3037445952693588e-11
DNE 70% ***** 2.590114420936441e-09
DNE 80% ***** 6.318031226241182e-10
DNE 90% ***** 7.623739294373904e-10
Biology
DNE 60% ***** 6.683843392564294e-12
DNE 70% ***** 2.0062657415147244e-10
DNE 80% ***** 4.219912819737888e-09
DNE 90% ***** 4.6264082989889694e-07
Built Environment & Design
DNE 60% ***** 7.293359814355617e-05
DNE 70% ***** 0.0007403695719125558
DNE 80% ***** 0.004854087599314228
DNE 90% ***** 0.009486319923078742
Clinical Medicine
DNE 60% ***** 3.3768433618889984e-06
DNE 70% ***** 6.675106678273537e-08
DNE 80% ***** 4.836198993260144e-08
DNE 90% ***** 1.963334219794505e-07
Earth & Environmental Sciences
DNE 60% ***** 2.2080311919639946e-17
DNE 70% ***** 9.426255110410718e-17
DNE 80% ***** 1.4581802843943654e-16
DNE 90% ***** 3.6104020619914153e-13
Economics & Business
DNE 60% ***** 8.418736012389252e-06
DNE 70% ***** 8.313513541711179e-07
DNE 80% ***** 4.261341134277354e-07
DNE 90% ***** 2.122206631531

In [ ]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    t_stat, p_value = stats.ttest_ind(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 1.8446539652636836e-15
DNE 70% ***** 2.452879440120457e-14
DNE 80% ***** 4.108006876900466e-16
DNE 90% ***** 3.207054104853677e-15
Europe
DNE 60% ***** 1.5842888439409607e-17
DNE 70% ***** 1.854399332815035e-16
DNE 80% ***** 1.9056462809294326e-17
DNE 90% ***** 9.881782250931794e-15
Middle East
DNE 60% ***** 1.7669579861347058e-13
DNE 70% ***** 1.6586892066612853e-12
DNE 80% ***** 1.1584235713458035e-10
DNE 90% ***** 5.720980337726858e-09
North Africa
DNE 60% ***** 2.8281726016028537e-09
DNE 70% ***** 1.3362579235352353e-08
DNE 80% ***** 2.8637300131242322e-05
DNE 90% ***** 0.0007014075584287091
North America
DNE 60% ***** 4.66665533731337e-09
DNE 70% ***** 3.482621341648002e-08
DNE 80% ***** 2.3987033888365097e-08
DNE 90% ***** 1.287327952581216e-07
Oceanic
DNE 60% ***** 2.43754254940892e-16
DNE 70% ***** 2.1356779890060427e-11
DNE 80% ***** 2.8457164764851084e-12
DNE 90% ***** 9.314997081307388e-11
South/Central America
DNE 60% ***** 2.1085824540442

## Mann-Whitney U Test

In [ ]:
for field in fields:
  print(field)
  sf = df[df["Field"] == field]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

Agriculture, Fisheries & Forestry
DNE 60% ***** 3.109601025095167e-09
DNE 70% ***** 1.3338356473921844e-08
DNE 80% ***** 3.382392386980057e-09
DNE 90% ***** 7.912288509126652e-09
Biology
DNE 60% ***** 8.691199596826435e-10
DNE 70% ***** 3.388001798692636e-10
DNE 80% ***** 7.950703918940211e-11
DNE 90% ***** 2.473382602375682e-09
Built Environment & Design
DNE 60% ***** 3.90303964513264e-05
DNE 70% ***** 7.863213072880285e-06
DNE 80% ***** 1.0299403064366306e-05
DNE 90% ***** 3.977965325094188e-05
Clinical Medicine
DNE 60% ***** 2.0260827985837603e-06
DNE 70% ***** 2.809785042045741e-07
DNE 80% ***** 8.401934428730703e-08
DNE 90% ***** 5.250085347210622e-08
Earth & Environmental Sciences
DNE 60% ***** 1.0384502858735705e-13
DNE 70% ***** 1.0638935047322592e-14
DNE 80% ***** 2.005033781173565e-16
DNE 90% ***** 3.8321982646469414e-14
Economics & Business
DNE 60% ***** 1.0461020848105665e-05
DNE 70% ***** 5.248452312805656e-09
DNE 80% ***** 2.1369857731991476e-10
DNE 90% ***** 7.1798146291

In [ ]:
for region in regions:
  print(region)
  sf = df[df["Region"] == region]
  for metric in metrics:
    highlyCited = sf[sf["Citation Level"] == "High"][metric].to_list()
    lowCited = sf[sf["Citation Level"] == "Low"][metric].to_list()
    u_stat, p_value = stats.mannwhitneyu(highlyCited, lowCited, alternative='greater')
    print(metric, "*****", p_value)

East/South-East Asia
DNE 60% ***** 3.0881606995873803e-12
DNE 70% ***** 2.9616533530335346e-12
DNE 80% ***** 3.814053180993884e-14
DNE 90% ***** 4.864065296705354e-14
Europe
DNE 60% ***** 1.0269884311493085e-14
DNE 70% ***** 5.688442726183725e-15
DNE 80% ***** 6.238712731733028e-18
DNE 90% ***** 7.3075628690532e-17
Middle East
DNE 60% ***** 7.02903009217583e-12
DNE 70% ***** 3.071963102520502e-12
DNE 80% ***** 1.179126845791081e-13
DNE 90% ***** 2.9754578338342008e-12
North Africa
DNE 60% ***** 1.540983435955837e-08
DNE 70% ***** 2.969490657831105e-10
DNE 80% ***** 2.4408662838797767e-08
DNE 90% ***** 3.6244881159714075e-07
North America
DNE 60% ***** 7.477668593775378e-08
DNE 70% ***** 3.536284217691656e-09
DNE 80% ***** 4.791802566268835e-11
DNE 90% ***** 2.677326334768297e-11
Oceanic
DNE 60% ***** 1.2573563160759937e-11
DNE 70% ***** 1.0716382287693191e-11
DNE 80% ***** 1.9111982060300985e-13
DNE 90% ***** 1.3516023319712342e-12
South/Central America
DNE 60% ***** 3.3067139498988236